# Explainability and Fairness Analysis

This notebook covers:
- Permutation Importance
- Partial Dependence Plot (PDP)
- LIME explanations
- Fairness metrics (High vs Low Amount groups)


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("../data/creditcard.csv")
X = data.drop("Class", axis=1)
y = data["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)
rf.fit(X_train_scaled, y_train)

## Permutation Importance

In [ ]:
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

perm = permutation_importance(
    rf,
    X_test_scaled,
    y_test,
    n_repeats=10,
    random_state=42
)
importances = perm.importances_mean

plt.figure(figsize=(8, 4))
plt.bar(range(len(importances)), importances)
plt.title("Permutation Importance (Random Forest)")
plt.xlabel("Feature index")
plt.ylabel("Importance")
plt.show()

## Partial Dependence Plot (PDP) and ICE for `Amount`

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

amount_index = X.columns.get_loc("Amount")

PartialDependenceDisplay.from_estimator(
    rf,
    X_test_scaled,
    [amount_index],
    kind="both"
)
plt.title("PDP and ICE for Amount")
plt.show()

## LIME local explanations

In [ ]:
from lime.lime_tabular import LimeTabularExplainer

explainer = LimeTabularExplainer(
    X_train_scaled,
    feature_names=X.columns,
    class_names=["Non-Fraud", "Fraud"],
    mode="classification"
)

exp = explainer.explain_instance(
    X_test_scaled[0],
    rf.predict_proba,
    num_features=10
)

exp.show_in_notebook()

## Fairness metrics — high vs low transaction amounts

In [ ]:
from sklearn.metrics import recall_score

median_amt = X_test["Amount"].median()
high_group = X_test["Amount"] > median_amt

recall_high = recall_score(y_test[high_group], rf.predict(X_test_scaled[high_group]))
recall_low = recall_score(y_test[~high_group], rf.predict(X_test_scaled[~high_group]))

print("High Amount Recall:", recall_high)
print("Low Amount Recall:", recall_low)
print("Recall gap (high - low):", recall_high - recall_low)